# PropWar Role-Change Validation ? Fold 1

**Execution status:** real 2018?2025 canonical data and executed 2021 Fold 1 artifacts. This notebook is an inspectable audit companion; it does not claim release readiness.

## tl;dr

- Canonical table: 57,928 rows; 20,727 rows in the 2018?2020 audit window; zero duplicate keys.
- Equal-volume baseline counts match in all 72 2021 family-weeks.
- No role family passes all point-gate diagnostics.
- Combined full-detector alert volume has a weekly median of 38 and range of 27?59, above the protocol maximum of 20.
- Partial-game exits cannot be identified reliably from the available schema, so public persistence claims remain blocked.

## Context & Methods

This notebook validates saved outputs produced by:

```powershell
python scripts/build_role_validation_dataset.py --seasons 2018-2025 --coverage-seasons 2017-2025 --cache-dir data/raw/role_validation --output-dir outputs/role_validation --config config/role_change_validation.yaml
python scripts/run_role_validation.py --input outputs/role_validation/canonical_player_week_role.csv.gz --config config/role_change_validation.yaml --output-dir outputs/role_validation/fold_1 --fold fold_1 --mode development
```

### Key Assumptions

- Definitions and release gates come from `ROLE_CHANGE_VALIDATION_PROTOCOL.md` and `config/role_change_validation.yaml`.
- Fold 1 parameters were selected only on 2018?2020; 2021 is the test season.
- Future outcomes stay in the alert season.
- The required partial-game flag is present but explicitly unreliable because no trustworthy exit source exists.

In [1]:
from pathlib import Path
import json
import pandas as pd
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'outputs' / 'role_validation'
FOLD = OUT / 'fold_1'
config = yaml.safe_load((ROOT / 'config' / 'role_change_validation.yaml').read_text(encoding='utf-8'))
manifest = json.loads((OUT / 'canonical_build_manifest.json').read_text(encoding='utf-8'))
manifest

{'seasons': [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
 'coverage_seasons': [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
 'canonical_rows': 57928,
 'audit_rows_2018_2020': 20727,
 'audit_passed': True,
 'canonical_path': 'outputs\\role_validation\\canonical_player_week_role.csv.gz',
 'canonical_sha256': '11d57cdde92238024da8afbedabe124da746250aa9a2026422014382ccc14e90',
 'protocol_sha256': 'b9fcc357e98388bb15c2d7ae853620f8ccd6c2e60e491a6cfcb990bbfbfcadbe',
 'locked_decisions_sha256': '57da1e3ebed077bd52709fb3331eb99e719c056e9e840d8c6913b512d7e4ba00'}

## Data

In [2]:
canonical = pd.read_csv(OUT / 'canonical_player_week_role.csv.gz', low_memory=False)
source_coverage = pd.read_csv(OUT / 'source_coverage_by_season.csv')
row_counts = pd.read_csv(OUT / 'canonical_row_counts_2018_2020.csv')
missingness = pd.read_csv(OUT / 'canonical_missingness_by_season.csv')
join_coverage = pd.read_csv(OUT / 'join_coverage.csv')
exclusions = pd.read_csv(OUT / 'exclusion_ledger.csv')
selected_parameters = pd.read_csv(FOLD / 'fold_1_selected_parameters.csv')
summary = pd.read_csv(FOLD / 'summary_2021.csv')
comparisons = pd.read_csv(FOLD / 'comparisons_2021.csv')
equal_volume = pd.read_csv(FOLD / 'equal_volume_verification_2021.csv')
weekly_counts = pd.read_csv(FOLD / 'weekly_alert_counts_2021.csv')
alerts = pd.read_csv(FOLD / 'alerts_2021.csv.gz', low_memory=False)

print({'canonical_rows': len(canonical), 'seasons': sorted(canonical.season.unique()), 'fold1_alert_rows': len(alerts)})

{'canonical_rows': 57928, 'seasons': [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)], 'fold1_alert_rows': 2868}


## Results

### 1. Validate canonical grain and 2018?2020 audit

In [3]:
key = config['data']['key_columns']
audit = canonical[canonical.season.between(2018, 2020)].copy()
required = config['data']['required_columns']
checks = {
    'audit_rows': len(audit),
    'duplicate_key_rows': int(audit.duplicated(key, keep=False).sum()),
    'required_null_cells': int(audit[required].isna().sum().sum()),
    'quality_pass_rate': float(audit.data_quality_pass.mean()),
    'excluded_rows': len(exclusions[exclusions.season.between(2018, 2020)]),
}
assert checks['duplicate_key_rows'] == 0
assert checks['required_null_cells'] == 0
checks

{'audit_rows': 20727,
 'duplicate_key_rows': 0,
 'required_null_cells': 0,
 'quality_pass_rate': 0.9799295604766729,
 'excluded_rows': 416}

In [4]:
display(source_coverage)
display(row_counts)
display(join_coverage[join_coverage.season.between(2018, 2021)])

,season,pbp_rows,pbp_games,schedule_games,regular_weeks,player_stat_rows,player_stats_game_id_missing_rate,roster_rows,snap_rows,injury_rows,scrimmage_plays,participation_play_coverage,carry_player_id_coverage,target_player_id_coverage,complete_schema_and_games
0,2017,45268,256,256,17,16786,0.0,49210,22876,4949,32315,0.999938,1.0,0.922291,True
1,2018,45120,256,256,17,16728,0.0,50113,22895,4961,32124,0.999938,1.0,0.909896,True
2,2019,45339,256,256,17,16663,0.0,49561,22884,5202,32402,1.000000,1.0,0.899895,True
3,2020,45406,256,256,17,16774,0.0,41972,23790,5414,32855,1.000000,1.0,0.907842,True
4,2021,47651,272,272,18,18128,0.0,44539,25271,5348,34344,1.000000,1.0,0.908026,True
5,2022,47157,271,271,18,17981,0.0,44059,25168,5450,34028,1.000000,1.0,0.896498,True
6,2023,47399,272,272,18,17806,0.0,43545,25329,5451,34237,1.000000,1.0,0.889352,True
7,2024,47274,272,272,18,18128,0.0,44473,25398,5954,33701,1.000000,1.0,0.892872,True
8,2025,46452,272,272,18,18539,0.0,44697,25395,5783,33241,1.000000,1.0,0.890706,True


,season,role_family,rows,players,qualifying_rows,data_quality_pass_rows
0,2018,rb_carry_share,1467,163,1415,1415
1,2018,rb_opportunity_share,1467,163,1415,1415
2,2018,te_target_share,1483,134,1438,1438
3,2018,wr_target_share,2360,231,2284,2284
4,2019,rb_carry_share,1509,160,1463,1463
5,2019,rb_opportunity_share,1509,160,1463,1463
6,2019,te_target_share,1409,136,1371,1371
7,2019,wr_target_share,2389,236,2328,2328
8,2020,rb_carry_share,1569,172,1569,1569
9,2020,rb_opportunity_share,1569,172,1569,1569


,season,join,rows,matched_rows,coverage_rate
0,2018,opportunity_to_identity,29956,29956,1.000000
1,2018,participating_player_to_identity,9511,9420,0.990432
2,2019,opportunity_to_identity,30027,30027,1.000000
3,2019,participating_player_to_identity,9454,9360,0.990057
4,2020,opportunity_to_identity,30597,30597,1.000000
5,2020,participating_player_to_identity,9700,9700,1.000000
6,2021,opportunity_to_identity,32052,32052,1.000000
7,2021,participating_player_to_identity,10227,10227,1.000000


### 2. Verify Fold 1 outputs and equal-volume baseline

In [5]:
assert set(alerts['season'].unique()) == {2021}
assert equal_volume['equal_volume'].all()
assert (equal_volume['min_count'] == equal_volume['max_count']).all()

recomputed = (
    alerts[alerts.persistent.notna()]
    .groupby(['role_family', 'method'])['persistent']
    .mean().rename('recomputed_precision').reset_index()
)
reconciled = summary.merge(recomputed, on=['role_family', 'method'], how='left')
reconciled['difference'] = reconciled['precision'] - reconciled['recomputed_precision']
assert reconciled['difference'].abs().max() < 1e-12

display(selected_parameters)
display(comparisons)
print({'equal_volume_family_weeks': len(equal_volume), 'mismatches': int((~equal_volume.equal_volume).sum())})

,role_family,baseline_window,min_baseline_games,min_abs_delta,development_alerts,development_evaluable_alerts,development_precision,development_naive_precision,development_precision_improvement,selection_minimum_evaluable_alerts
0,rb_carry_share,4,4,0.15,610,474,0.622363,0.478992,0.143371,25
1,rb_opportunity_share,4,4,0.15,686,534,0.629213,0.516729,0.112485,25
2,wr_target_share,4,3,0.12,209,170,0.494118,0.252874,0.241244,25
3,te_target_share,6,4,0.11,56,52,0.500000,0.326531,0.173469,25


,role_family,full_alerts,naive_alerts,full_evaluable_alerts,naive_evaluable_alerts,full_precision,naive_precision,precision_improvement,relative_precision_improvement,precision_improvement_ci_low,precision_improvement_ci_high,full_reversion_rate,naive_reversion_rate,reversion_improvement,full_median_retention,naive_median_retention
0,rb_carry_share,273,273,222,221,0.572072,0.493213,0.078859,0.159889,0.021341,0.140606,0.281513,0.345833,0.064321,0.657328,0.467145
1,rb_opportunity_share,324,324,256,257,0.585938,0.517510,0.068428,0.132225,0.001807,0.143318,0.301418,0.322695,0.021277,0.642635,0.547792
2,te_target_share,35,35,25,26,0.200000,0.269231,-0.069231,-0.257143,-0.204585,0.066692,0.500000,0.454545,-0.045455,0.165951,0.165085
3,wr_target_share,85,85,59,61,0.440678,0.295082,0.145596,0.493409,0.015140,0.272534,0.472973,0.534247,0.061274,0.400202,0.298837


{'equal_volume_family_weeks': 72, 'mismatches': 0}


### 3. Check alert volume and point-gate diagnostics

In [6]:
full_weekly = weekly_counts[weekly_counts.method.eq('full_propwar')]
feed_volume = full_weekly.groupby('week').alerts.sum()
gate_diagnostic = pd.read_csv(FOLD / 'fold_1_gate_diagnostic.csv')
display(gate_diagnostic[['role_family','full_alerts','full_precision','precision_improvement','full_reversion_rate','reversion_improvement','full_median_retention','all_point_gates_pass']])
print({'weekly_mean': feed_volume.mean(), 'weekly_median': feed_volume.median(), 'weekly_min': feed_volume.min(), 'weekly_max': feed_volume.max()})
assert not gate_diagnostic['all_point_gates_pass'].any()
assert feed_volume.max() > 20

,role_family,full_alerts,full_precision,precision_improvement,full_reversion_rate,reversion_improvement,full_median_retention,all_point_gates_pass
0,rb_carry_share,273,0.572072,0.078859,0.281513,0.064321,0.657328,False
1,rb_opportunity_share,324,0.585938,0.068428,0.301418,0.021277,0.642635,False
2,te_target_share,35,0.200000,-0.069231,0.500000,-0.045455,0.165951,False
3,wr_target_share,85,0.440678,0.145596,0.472973,0.061274,0.400202,False


{'weekly_mean': np.float64(39.833333333333336), 'weekly_median': np.float64(38.0), 'weekly_min': np.int64(27), 'weekly_max': np.int64(59)}


## Takeaways

Fold 1 is reproducible and the equal-volume comparison is verified, but the evidence is negative:

- RB families are directionally better than naive but miss precision, improvement, and reversion gates.
- WR improves versus naive but remains too imprecise and reverts too often.
- TE performs worse than naive with limited evidence.
- Full-detector volume is far above the product constraint.
- Partial-game exclusion remains an unresolved high-severity data limitation.

The correct status is **Needs revision**. No public automated persistence claim is supported.

In [7]:
result_status = {
    'data_mode': 'REAL_DATA',
    'audit_passed_structurally': True,
    'equal_volume_verified': bool(equal_volume.equal_volume.all()),
    'families_passing_all_point_gates': int(pd.read_csv(FOLD / 'fold_1_gate_diagnostic.csv').all_point_gates_pass.sum()),
    'public_detector_claim_supported': False,
    'overall_assessment': 'Needs revision',
}
result_status

{'data_mode': 'REAL_DATA',
 'audit_passed_structurally': True,
 'equal_volume_verified': True,
 'families_passing_all_point_gates': 0,
 'public_detector_claim_supported': False,
 'overall_assessment': 'Needs revision'}